# 第 13 课 · 再多一块钱值不值（浏览器版）

配套课文：[docs/m0-foundations/13-derivatives-and-error-tails.md](../docs/m0-foundations/13-derivatives-and-error-tails.md)。

**怎么用**：本地装了 Jupyter 就直接打开这一页；没有环境就上 https://jupyter.org/try （JupyterLite）或 Colab，把这份 notebook 整份放进去，每一格按 `Shift + Enter` 运行。只用 Python 标准库，不需要安装任何东西。

每格跑出来的数字和课文里印的逐字一致（种子钉在 7）。有两格要跑一万遍乘 250 天、以及最多二十万遍的模拟，会多等几秒——次数是钉死的，别改，改了数字就对不上了。


## 一、你手上有一个真实的困惑

第一个困惑是定价或者投放。你的套餐从 32 涨到 33，一天是多赚还是少赚？你其实隐约知道答案不是"是／否"，而是"得看现在定在几块"。但你没法把它算出来：你知道涨价销量会掉，掉多少说不清，于是这件事一直停在"感觉再涨就要亏"。

第二个困惑来自你前面几节课。第 07、08 课告诉你抽签抽得少会有毛边，第 12 课告诉你把已经知道的事挑出来不抽能少掉一半。但你一直没问过第三件事：**我为了好算而把算法简化了，简化本身错了多少？** 每天乘一遍太麻烦，直接乘 1.12 省事——这个省事要收你多少利息？

这两个困惑是同一种数学的两面，也是原书后面几条主线共用的语言。第 7 章算"股价动一点，期权价格动多少"用的是第一面的语言；第 6 章比较"一口气走完"和"切成一百步走完"哪个更准，用的是第二面。这一节课把两面都落到你今天的数字上。


## 二、"多一单位值多少钱"是一个数，而它一直在变

场景是校门口那杯柠檬茶，硬成本 8 块。定价 $p$，一天卖 $D(p)$ 杯。你手上只有两条记录：卖 10 块一天出 60 杯，试过 11 块出 54 杯。于是你把这条需求写成直线：

$$D(p) = 120 - 6p$$

一天的利润是

$$L(p) = (p - 8) \times (120 - 6p)$$

这个式子替你回答的问题只有一个：**定这个价，一天赚几块。** 先搭好三台机器（`profit` 算利润；`forward` 和 `central` 是第四节的两种读法——注意它们收的第一个东西是**函数本身**，`profit` 后面不带括号，意思是"把这台机器交出去"。这一格只是定义，运行它不会打印任何东西）：


In [1]:
import math
import random

COST = 8.0        # 一杯的硬成本
DEMAND_A = 120.0  # 定 1 元时一天理论能卖多少杯
DEMAND_B = 6.0    # 每涨 1 元少卖几杯


def demand(p):
    return DEMAND_A - DEMAND_B * p


def profit(p):
    """一天卖 p 元一杯，赚多少钱。"""
    return (p - COST) * demand(p)


def spread(vals):
    n = len(vals)
    m = sum(vals) / n
    return math.sqrt(sum((x - m) ** 2 for x in vals) / (n - 1))


def forward(f, x, h):
    """往前挪一步读：把 [x, x+h] 这段的平均变化当成 x 这一点的答案。"""
    return (f(x + h) - f(x)) / h


def central(f, x, h):
    """前后各挪一步再平均：把 x 两侧的信息都用上。"""
    return (f(x + h) - f(x - h)) / (2 * h)


然后把它一档一档列出来（实测）：


In [2]:
print(f"一杯硬成本 {COST:.0f} 元；定 10 元一天卖 {demand(10):.0f} 杯，每涨 1 元少卖 {DEMAND_B:.0f} 杯")
print("定价      一天利润    涨 1 块之后    多一块值")
for p in [9, 10, 11, 12, 13, 14, 15, 16]:
    print(f"{p:>4} 元  {profit(p):>10.0f}  {profit(p + 1):>12.0f}  {profit(p + 1) - profit(p):>+11.0f}")


一杯硬成本 8 元；定 10 元一天卖 60 杯，每涨 1 元少卖 6 杯
定价      一天利润    涨 1 块之后    多一块值
   9 元          66           120          +54
  10 元         120           162          +42
  11 元         162           192          +30
  12 元         192           210          +18
  13 元         210           216           +6
  14 元         216           210           -6
  15 元         210           192          -18
  16 元         192           162          -30


这张表就是"值不值"的全部真相：**同样是多涨一块，在 10 元那档值 42 块，在 14 元那档倒亏 6 块。** 所以你真正要问的从来不是"涨一块好不好"，而是"我现在在 13 这一块，往前一格好不好"。

那个"在这一档，多一单位值多少钱"就叫**导数**，经济学者嘴里叫**边际**。它替你回答的问题：**在我现在这个位置上，再往前一小步能换回多少。**

你现在就可以把第一个结论拿走，它不需要任何符号：**"值多少"不是一个数，是随你所在位置变化的一整条。** 任何人给你一个"敏感度"却不告诉你"在哪个位置量的"，他给的信息就不完整。外卖平台的"客单价弹性"、运营口中的"投放边际回报"、朋友的"这基金对利率很敏感"，全适用。


## 三、"多一块值 0"的那一档，就是你该待的地方

以 0.1 元为一档，从 8 块一路试到卖光（枚举，不解方程）：


In [3]:
best_p, best_v = None, -1.0
p = 8.0
while p <= DEMAND_A / DEMAND_B:
    if profit(p) > best_v:
        best_v, best_p = profit(p), p
    p += 0.1
print(f"以 0.1 元为档，从 {COST:.0f} 元一档档试上去：最赚的是 {best_p:.1f} 元，一天 {best_v:.0f} 块")
print("手算：利润 = (p−8)(120−6p)，展开是 −6p² + 168p − 960；令导数 168 − 12p = 0 → p = 14")
print(f"核对（步子缩到 0.1 元来读）：p=13.9 时多一角值 {forward(profit, 13.9, 0.1):+.1f}，p=14.1 时值 {forward(profit, 14.1, 0.1):+.1f}")
print(f"过了顶点每多一角都是负的：p=16 → {forward(profit, 16, 0.1):+.1f}")


以 0.1 元为档，从 8 元一档档试上去：最赚的是 14.0 元，一天 216 块
手算：利润 = (p−8)(120−6p)，展开是 −6p² + 168p − 960；令导数 168 − 12p = 0 → p = 14
核对（步子缩到 0.1 元来读）：p=13.9 时多一角值 +0.6，p=14.1 时值 -1.8
过了顶点每多一角都是负的：p=16 → -24.6


为什么"导数等于零"的地方就是最赚？看实测的两个数就明白了：13.9 那一档再涨一角还值 **+0.6**，到 14.1 同样涨一角变成 **−1.8**。中间必然经过一个"再涨一角刚好不赚不赔"的位置；过了它，每往前一步都是负的。所以**顶点不是"值最多"的地方，是"值零"的地方**——你一直往前加，加到那一单位不再划算为止，那一刻就是最优。

这就是所有"该投到哪一档"的共同手续：把结果写成一件事的函数，问"再多一单位还值不值"，一直加到它值为零。折扣力度、投放预算、加班时数、备货量、一条视频里塞几个标签，全是这个形状。

一条必须给的诚实话：这个手续只在"结果真的是一条光滑下弯的曲线"时好使。现实里你的档位常常是整数（班次、篇数），或者曲线根本不止一个鼓包——平台在不同价位段的流量是断崖式的，你从 29 涨到 30 会掉进另一个流量池。这种情况下"令导数为零"会给你一个漂亮而错的答案，而第二节那张一档一档列完的表反而老实。**枚举永远是底牌，导数只是抄近路；近路的代价是它假设曲线光滑。**


## 四、你读出来的那个"边际"是歪的，歪多少看曲线弯多厉害

第二节的表里藏着一件事。还是 p=10 这一档，用三种步子去读"多一块值多少"：


In [4]:
print("把步子缩小看瞬时，还是 p=10 那一档：")
print(f"一步读（涨满 1 块）      ：{(profit(11) - profit(10)) / 1:.1f}")
print(f"小步读（涨 0.1 块）      ：{forward(profit, 10, 0.1):.1f}")
print(f"两侧平均读（9.9 到 10.1）：{central(profit, 10, 0.1):.1f}")
print(f"手算的瞬时值 168 − 12p 在 p=10：{168 - 12 * 10:.1f}")


把步子缩小看瞬时，还是 p=10 那一档：
一步读（涨满 1 块）      ：42.0
小步读（涨 0.1 块）      ：47.4
两侧平均读（9.9 到 10.1）：48.0
手算的瞬时值 168 − 12p 在 p=10：48.0


三个数都在回答"多一块值多少"，也都对，只是答的不是同一个问题：42 是 **10 到 11 这一段**的平均，47.4 是 **10 到 10.1 这一段**的平均，48.0 是 **9.9 到 10.1 这一段**的平均——而它的中点正好是 10。

利润是条下弯的曲线（涨得越高，每一块越不值），所以你往哪边迈一步，读数就被哪边拽走。这里有一个可以直接验证的规律：


In [5]:
print(f"在 p=10 涨满 1 块，读到 {(profit(11) - profit(10)):.0f}；")
print(f"而 10 那一点的瞬时值是 {central(profit, 10, 0.001):.1f}，11 那一点是 {central(profit, 11, 0.001):.1f}。")
print(f"答案在中间：10.5 那一点的瞬时值正好是 {central(profit, 10.5, 0.001):.1f}。")


在 p=10 涨满 1 块，读到 42；
而 10 那一点的瞬时值是 48.0，11 那一点是 36.0。
答案在中间：10.5 那一点的瞬时值正好是 42.0。


**一步读出来的数，其实是这一段中点上的瞬时值**，不是出发点的。这不是四舍五入级别的小毛病。等你手上只有按天记录的数据，"这个月的边际"真正属于的是这个月和上个月的中点；你把 42 报成"10 元这一档值多少"，就悄悄把它挪了半格。

那怎么把读数读准？办法是把两侧都用上。换一条弯得更厉害的曲线（$e^x$，在 $x = 2$ 处真导数 $= 7.3891$），试五种步长：


In [6]:
print("步长      往前读        误差        两侧平均读      误差")
true_d = math.exp(2)
for h in [1.0, 0.5, 0.25, 0.125, 0.0625]:
    f_err = abs(forward(math.exp, 2, h) - true_d) / true_d * 100
    c_err = abs(central(math.exp, 2, h) - true_d) / true_d * 100
    print(f"{h:<8.4f}  {forward(math.exp, 2, h):>8.4f}  {f_err:>8.1f}%   {central(math.exp, 2, h):>9.4f}  {c_err:>7.2f}%")


步长      往前读        误差        两侧平均读      误差
1.0000     12.6965      71.8%      8.6836    17.52%
0.5000      9.5869      29.7%      7.7008     4.22%
0.2500      8.3947      13.6%      7.4663     1.04%
0.1250      7.8707       6.5%      7.4083     0.26%
0.0625      7.6249       3.2%      7.3939     0.07%


规律很干净：**步长减半，往前读的错减半，两侧平均读的错降到四分之一。** 前者叫一阶准，后者叫二阶准。多算一次调用（往前挪一格不如前后各挪一格），换回来的是阶数。

原书第 7 章算期权的敏感度（Greeks）时，这是正式议题而不是小技巧：同一份模拟，往前扰动和两侧扰动给出的数误差量级不同，而误差量级直接决定那个数字能不能报出口。你现在已经能读懂那一节在争什么了。


## 五、泰勒展开只回答一个问题：我省掉的那一项，错多少

$$e^x \approx 1 + x + \frac{x^2}{2} + \frac{x^3}{6}$$

它替你回答的问题：**我用左边这个好算的式子代替右边那个难算的，会错多少？** 答案只有一句：**会错多少，约等于你扔掉的第一项有多大。**

这张表就是这句话的证据（实测）：


In [7]:
print("x        真值        1+x       差        1+x+x²/2      差")
for x in [0.01, 0.04, 0.1, 0.3, 0.5, 1.0]:
    truth = math.exp(x)
    one = 1 + x
    two = 1 + x + x * x / 2
    print(f"{x:>6.2f}  {truth:>10.6f}  {one:>8.4f}  {truth - one:>+9.6f}   {two:>10.6f}  {truth - two:>+10.6f}")
print(f"x=0.04 那一行：扔掉的第一项是 x²/2 = {0.04 ** 2 / 2:.6f}，实测误差 {math.exp(0.04) - 1.04:.6f}。")


x        真值        1+x       差        1+x+x²/2      差
  0.01    1.010050    1.0100  +0.000050     1.010050   +0.000000
  0.04    1.040811    1.0400  +0.000811     1.040800   +0.000011
  0.10    1.105171    1.1000  +0.005171     1.105000   +0.000171
  0.30    1.349859    1.3000  +0.049859     1.345000   +0.004859
  0.50    1.648721    1.5000  +0.148721     1.625000   +0.023721
  1.00    2.718282    2.0000  +0.718282     2.500000   +0.218282
x=0.04 那一行：扔掉的第一项是 x²/2 = 0.000800，实测误差 0.000811。


看 x=0.04 那一行：扔掉的第一项是 $x^2/2 = 0.000800$，实测误差 $0.000811$。看 x=0.1 那一行：1+x 错 0.00517，而 $x^2/2 = 0.00500$；1+x+x²/2 错 0.000171，而下一项 $x^3/6 = 0.000167$。**每一档的误差都被"你扔掉的第一项"报准了。**

读这张表的方式不是背式子，是看"差"那一列什么时候变得你不能忍。$x = 0.01$ 时一阶近似错十万分之五；$x = 0.3$ 时已经差到 0.05，占真值 3.7%；$x = 1$ 时一阶近似报 2、真值 2.72，错掉 26%。

**"能不能用省事式子"从来没有全局答案，只有这一行一行的答案。** 你要做的只有一件事：先把准备扔掉的第一项算出来，看它多大，再决定扔不扔。

同一个动作管另一件你常干的事。把 $e^x$ 那行换个方向写：

$$\ln(1 + r) \approx r - \frac{r^2}{2}$$

核对一下（$r = 0.1$）：


In [8]:
print(f"r=0.1：真值 {math.log(1.1):.6f}，r − r²/2 = {0.1 - 0.1 ** 2 / 2:.6f}，差 {math.log(1.1) - (0.1 - 0.1 ** 2 / 2):+.6f}（就是下一项 r³/3 = {0.1 ** 3 / 3:.6f}）")


r=0.1：真值 0.095310，r − r²/2 = 0.095000，差 +0.000310（就是下一项 r³/3 = 0.000333）


真值 0.095310，两项式给 0.095000，差 0.000310，而下一项 $r^3/3 = 0.000333$——又是它。别小看这一行，第六节那个"一半人亏钱"的结论全部来自它的第二项。


## 六、第二项决定你属于哪一半

一笔钱 100，投 250 天，每天涨跌的平均是 0（一个真正不赚不赔的赌局），单日起伏 10%。

按一阶想：每天乘的那个数平均是 1，250 天乘下来还是 1，结论是"不亏不赚"。实测（一万遍）：


In [9]:
random.seed(7)
ends = []
for _ in range(10000):
    v = 100.0
    for _ in range(250):
        v *= 1 + random.gauss(0, 0.10)
    ends.append(v)
ends.sort()
med = ends[len(ends) // 2]
below = sum(1 for e in ends if e < 100) / len(ends) * 100
theory = 100 * math.exp(-0.10 ** 2 * 250 / 2)
print(f"实测（一万遍）：平均 {sum(ends) / len(ends):.1f}，中位 {med:.1f}，{below:.1f}% 的路径低于起点")
print(f"二阶项给的预测：中位 ≈ 100 × e^(−250 × 0.10²/2) = 100 × e^(−1.25) = {theory:.1f}")
print(f"平均那一行的诚实话：真值是 100，实测 {sum(ends) / len(ends):.1f}，因为这个平均自己的毛边就有 ±{spread(ends) / math.sqrt(len(ends)):.1f}（少数路径能翻好几倍，第 08 课那个厚尾毛病）")


实测（一万遍）：平均 97.3，中位 28.4，78.9% 的路径低于起点
二阶项给的预测：中位 ≈ 100 × e^(−250 × 0.10²/2) = 100 × e^(−1.25) = 28.7
平均那一行的诚实话：真值是 100，实测 97.3，因为这个平均自己的毛边就有 ±2.8（少数路径能翻好几倍，第 08 课那个厚尾毛病）


一阶说的确实是那个平均——它的真值就是 100，只是这个平均自己的毛边就有 ±2.8。而一半的人落在中位 **28.4** 上，本金剩不到三成。

差在哪？就在第五节那行 $\ln(1+r) \approx r - r^2/2$。250 天把每天那一项加起来，$r$ 那一项平均是 0，$r^2/2$ 那一项平均是 $\sigma^2/2 = 0.005$，天天照扣：

$$\text{中位} \approx 100 \times e^{-250 \times 0.10^2 / 2} = 100 \times e^{-1.25} = 28.7$$

公式给 28.7，实测 28.4。**第 10 课那个"波动拖累 $\sigma^2/2$"就是这里的第二项。** 那时它是背下来的，现在你知道它从哪来：它是 $\ln(1+r)$ 展开的第二项，是你随手扔掉就会算错一半人的那一块。

现实里这条最贵。一个日波动 10%、日均 0 的产品，宣传口径写"长期期望不亏"，这句没错；它的一半用户本金只剩三成，这句也没错。两句话之间的差就是那一个 $-\sigma^2 T/2$。所以你下次听到"这个平均收益是正的所以放心"，条件反射应该是：**这个平均是给谁算的？中位那一半人在哪？**


## 七、两种误差，只有一种能用算力买掉

回到最省事的那个算法。30 天，每天乘 $(1 + 0.4\%)$。精确算：$100 \times 1.004^{30} = 112.72$。偷懒算（当成一步涨 12%）：$100 \times 1.12 = 112.00$。**省事省掉 0.72 块，而这 0.72 正是第五节里 $x^2/2$ 那一块：$0.12^2/2 = 0.0072$，乘本金 100。**

这条偏差有多硬？把 30 天切成几段，每段用同样的偷懒法：


In [10]:
MU, SD, DAYS = 0.004, 0.03, 30
print("模型：每天乘 (1 + 平均 0.4% 起伏 3%)，看 30 天后终值的平均。")
truth_exact = 100 * (1 + MU) ** DAYS
one_step = 100 * (1 + DAYS * MU)
bias_one = one_step - truth_exact
print(f"真值（精确算法）：{truth_exact:.2f}")
print(f"一步近似（偷懒当成一步涨 12%）：{one_step:.2f}  偏差 {bias_one:+.2f}（确定性的，跑多少遍都不会动）")
print("把 30 天切成几段，每一段都用同样的偷懒法：")
for m in [1, 2, 4, 8, 16]:
    seg = 1 + (DAYS / m) * MU
    v = 100 * seg ** m
    print(f"切 {m:>2} 段：{v:>7.2f}  偏差 {v - truth_exact:+.3f}")
print("段数翻倍、偏差减半——一阶准。把第二项加回去（1 + x + x²/2，x=0.12）：")
two_order = 100 * (1 + DAYS * MU + (DAYS * MU) ** 2 / 2)
print(f"  {two_order:.2f}  偏差 {two_order - truth_exact:+.3f}——一次修正顶得上切几十段。")


模型：每天乘 (1 + 平均 0.4% 起伏 3%)，看 30 天后终值的平均。
真值（精确算法）：112.72
一步近似（偷懒当成一步涨 12%）：112.00  偏差 -0.72（确定性的，跑多少遍都不会动）
把 30 天切成几段，每一段都用同样的偷懒法：
切  1 段： 112.00  偏差 -0.723
切  2 段： 112.36  偏差 -0.363
切  4 段： 112.55  偏差 -0.172
切  8 段： 112.65  偏差 -0.073
切 16 段： 112.70  偏差 -0.023
段数翻倍、偏差减半——一阶准。把第二项加回去（1 + x + x²/2，x=0.12）：
  112.72  偏差 -0.003——一次修正顶得上切几十段。


段数翻倍、偏差减半——和第四节那个"往前读"一样是一阶准。而把第二项加回去（$1 + x + x^2/2$），偏差从 0.723 掉到 **0.003**：一次修正顶得上切一百段。原书第 6 章第二节整章（二阶方法）做的就是把这一类修正系统化——**不是把步子切细，而是找到被扔掉的那一项，把它加回去。**

再把它跑成模拟，看另一种误差怎么动。要估的是 30 天后终值的平均，真值 112.72：


In [11]:
print("毛边随次数怎么变（真值同上）：")
for runs in [2000, 20000, 200000]:
    random.seed(7)
    vals = []
    for _ in range(runs):
        v = 100.0
        for _ in range(DAYS):
            v *= 1 + random.gauss(MU, SD)
        vals.append(v)
    est = sum(vals) / len(vals)
    se = spread(vals) / math.sqrt(runs)
    print(f"{runs:>6} 遍：估计 {est:.2f} ± {se:.2f}（毛边）；近似偏差仍是 {bias_one:+.2f}")
print(f"二十万遍时毛边已经比偏差小：此后总误差由那个 {abs(bias_one):.2f} 说了算，多加算力白加。")


毛边随次数怎么变（真值同上）：
  2000 遍：估计 112.72 ± 0.41（毛边）；近似偏差仍是 -0.72
 20000 遍：估计 112.80 ± 0.13（毛边）；近似偏差仍是 -0.72
200000 遍：估计 112.68 ± 0.04（毛边）；近似偏差仍是 -0.72
二十万遍时毛边已经比偏差小：此后总误差由那个 0.72 说了算，多加算力白加。


**毛边按 $\sqrt{\text{次数}}$ 缩，偏差纹丝不动。** 到二十万遍，毛边已经比偏差小一个量级，你的总误差由那 0.72 说了算，再多的算力买不回它。

下面这张图把今天两件事画在同一个框里。左边是三根误差棒（同一个模型，只加次数），右边就是上面那张切段表（同一个偷懒法，只切段），最右边那根几乎看不见的橙条，是把 $x^2/2$ 加回去之后剩下的偏差：

![毛边越跑越小，偏差一步不让](https://raw.githubusercontent.com/JavenZheng/monte-carlo-finance/main/assets/fig13_bias_vs_noise.svg)

图由 [code/fig13_bias_vs_noise.py](../code/fig13_bias_vs_noise.py) 生成，用的就是上面这一轮抽签（种子 7）。看左边那根橙线：它从头到尾没动过一格。

这正是第 12 课练习四那个人的病：他跑五百万遍，压的是毛边；他的错出在"用三条记录定了一个组的平均"，那是偏差。今天可以把它推广成一条你随身带的判据：

**把次数加到十倍，看误差降到几分之几。降到三分之一，说明起作用的是毛边，该买算力；几乎不动，说明起作用的是偏差，该改算法、补数据或者换口径。**

这也是原书把第 4 章（方差缩减）和第 6 章（离散化）分成两章的原因：前者治毛边，后者治偏差，两笔预算不能互抵。


## 八、今晚就能用起来

代码在 [code/lesson13_derivatives_and_error_terms.py](../code/lesson13_derivatives_and_error_terms.py)。改成你自己的场景只动最上面三个数：`COST`（硬成本）、`DEMAND_A`（定 1 元时的量）、`DEMAND_B`（每涨 1 单位少多少）。

**第一件：给你的价目表、折扣或者投放预算做一次"多一块值多少"的表。** 第二节的表只有三列：定这个价的结果、往前一档的结果、两者的差。你只需要两个历史点就能定出那条线——"卖 32 出 60 单、卖 33 出 54 单"这种话你本来就说得出。跑完找那一档从正变负的位置，那就是你该待的档位。**记得第三节那条诚实话：档位是整数、或者中间有断崖，就把表列完，别解方程。**

**第二件：别人给你一个敏感度数字时，问三句。** "在哪一点上量的？""步子多大？""往前挪还是两侧平均？"这三句分别对应第二、四节。你不需要懂金融，只要对方答不出第三句，他就多半是往前读了一格、然后把中点的数安在了起点上。

**第三件：报自己的模拟结果时，把两种误差写成两行。** 一行毛边（±0.04，来自抽签次数），一行偏差（−0.72，来自你简化那一步、你历史数据的条数、你钉死的那些假设）。两行的处置办法完全不同：前者加钱，后者改算法。合成一行报出去，别人只会回你一句"再多跑点数据"。


## 这一课带走的一句话

多一单位值多少，要看你现在在第几单位，值到零那一档就是最划算的那档；而你为省事扔掉的那一步不是免费的——错的量就是你扔掉的第一项，它和抽签的毛边是两回事，加多少遍模拟都修不掉。

配套课文：[第 13 课（网页版）](../docs/m0-foundations/13-derivatives-and-error-tails.md)


---

**许可**：正文 CC BY-NC-SA 4.0，代码 MIT，见 [LICENSE.md](../LICENSE.md)。

**建议购买正版**：Paul Glasserman, *Monte Carlo Methods in Financial Engineering*, Springer, 2003；中译本《金融工程中的蒙特卡罗方法》。本 notebook 未复制原书任何文字、图表或表格。
